# BiRefNet Training Data Preparation (Colab)

This notebook runs the **Phase 2** data pipeline of the `my-bg-remover` project
**end to end** on Google Colab: it downloads the training sources (DIS5K, CAMO,
COD10K, P3M-10k, Transparent-460, ...) at full size, produces transparency/camouflage-weighted
composite images, and converts the result into the `im/` + `gt/` folder layout
expected by BiRefNet's official training code, saving it to Google Drive.

**Why on Colab?** The local development environment's disk budget is very tight
(~15GB free); therefore only small *validation samples* (`data/raw_train/`,
<=300MB per source) were downloaded locally and the tools (scripts) were tested there.
Colab's disk space (~200GB on a T4/A100 runtime) is sufficient to download and
process the full datasets.

## What does this notebook do? (summary for non-ML readers)

1. **Mount Drive** — we connect your Google Drive to Colab so the large data is not
   lost on Colab's temporary disk but written persistently to Drive.
2. **Fetch the code repository (repo)** — we bring the project's scripts
   (`build_trainset.py`, `make_composites.py`, `export_birefnet.py`) to Colab.
3. **Download the raw data** — we download the training images and their
   "correct answer" masks (ground truth / GT — black-and-white images that show
   pixel by pixel exactly where the object is) from the internet (Hugging Face, Google Drive).
4. **Build the manifest** — we produce an "inventory" (manifest) file listing which
   image belongs to which category (hair, transparent, camouflage, ...) and where
   its GT file is located.
5. **Generate composites** — we paste objects onto different backgrounds (compositing)
   and add light distortions (augmentation: color/brightness shifts, JPEG
   degradation, blur, mirroring) so the model learns from more varied examples.
   The transparency and camouflage categories are represented with more copies, in
   line with the project's goal (see cell (e)).
6. **Export to BiRefNet format** — the model to be trained (BiRefNet) expects a
   specific folder layout (`im/` = images, `gt/` = masks, pairs matched by identical
   file names). In the final step we convert the data into this layout.
7. **Copy to Drive + verify** — we copy the result to Drive for persistence and
   check that the file counts are consistent.

**This notebook was written WITHOUT being executed** (the development environment
has no Colab/GPU/Drive access) — each cell was designed to be self-contained and
documented; simply run them in order on Colab. If a cell fails, read the error
(it is usually a missing Drive file or a changed HF repo path), fix the relevant
parameter, and continue from that cell — the scripts were designed to be idempotent
(they skip existing files), so you never need to start over.

## Parameters

You can change the values in this cell to suit your needs. Fill in **only one** of
the two repo sources: `REPO_GIT_URL` (if you pushed the project to GitHub) or
`REPO_ZIP_ON_DRIVE` (if you uploaded the repo to Drive as a zip — a pragmatic
fallback since in this environment the repo may not have a GitHub URL yet).

In [ ]:
# --- Repo source: fill in ONLY ONE of the two ---
REPO_GIT_URL = ""  # e.g. "https://github.com/<user>/my-bg-remover.git"
REPO_ZIP_ON_DRIVE = ""  # e.g. "/content/drive/MyDrive/my-bg-remover.zip" (if there is no git clone)

# --- Working directories ---
DRIVE_ROOT = "/content/drive/MyDrive"
WORKDIR = "/content/my-bg-remover"          # where the repo is extracted onto Colab's temporary disk
DRIVE_OUTPUT_SUBDIR = "bg-remover-data"     # name of the folder on Drive where the result is written

# --- Data parameters (by default kept the SAME as scripts/make_composites.py —
#     drift prevention; fill in CATEGORY_MULTIPLIERS only if you EXPLICITLY want a
#     different value here, see the line below) ---
SEED = 42
CATEGORY_MULTIPLIERS = None  # None = use the scripts/make_composites.py default
# (currently transparent×10, camouflage×2); if a non-empty dict is given, cell (e) overrides with it
TARGET_TOTAL_IMAGES = 40000  # approximate target total composite count (per-image is derived from this)
BG_POOL_SIZE = 3000          # background pool size (from BG-20k; the full 15k is possible but a subset saves disk/time)

# --- Optional sources to download from Google Drive ---
# COD10K-TR is always attempted (its drive_id is present in data/train_sources.json,
# important for the camouflage goal). HIM2K/AM-2k contribute to the "general"
# category but may require Drive access/a consent form; if they cannot be found/
# downloaded the pipeline does NOT break (the cells below tolerate this via except)
# — so they are attempted by default but can be turned off as OPTIONAL.
DOWNLOAD_OPTIONAL_DRIVE_SOURCES = True  # attempt HIM2K + AM-2k? (False -> skip)

print("Parameters set. Make sure you filled in either REPO_GIT_URL or REPO_ZIP_ON_DRIVE.")
assert REPO_GIT_URL or REPO_ZIP_ON_DRIVE, "One of REPO_GIT_URL or REPO_ZIP_ON_DRIVE must be filled in"

# --- HF download environment settings (lesson from a stuck download): large
# parquet/zip shards can sometimes hang in HfFileSystem/hf_hub_download with the
# default timeout (see the task report) — we raise HF_HUB_DOWNLOAD_TIMEOUT.
# HF_TOKEN is picked up automatically if defined in Colab Secrets (under the name
# "HF_TOKEN") (useful for rate limits/gated repo access; if not defined it is
# silently skipped, most sources work with anonymous access).
import os as _os

_os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")
try:
    from google.colab import userdata as _userdata

    _hf_token = _userdata.get("HF_TOKEN")
    if _hf_token:
        _os.environ["HF_TOKEN"] = _hf_token
        print("HF_TOKEN retrieved from Colab Secrets.")
except Exception as _e:
    print(f"Could not retrieve HF_TOKEN (not in Secrets/no access, continuing with anonymous access): {_e}")

## (a) Mount Google Drive

Mounting Drive tells Colab "use this folder as if it were my Google Drive". This
way the data we download/produce is NOT deleted when the Colab session ends
(usually after a few hours) — it stays persistently on Drive. When you run this,
Colab will ask for a Google account permission; approve it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os
os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/{DRIVE_OUTPUT_SUBDIR}", exist_ok=True)
print(f"Drive mounted. Working directory: {WORKDIR}")
print(f"Output will be written to Drive at: {DRIVE_ROOT}/{DRIVE_OUTPUT_SUBDIR}")

## (b) Fetch the Repo + Install Dependencies

We bring the project's code (the scripts, the `pyproject.toml` dependency list) to
Colab. `uv` (the fast package manager used in local development) is not installed
on Colab — instead a standard `pip install -e .` is sufficient: it installs the
`dependencies` list from `pyproject.toml` (torch, transformers, pillow, numpy,
huggingface-hub, ...) and adds the project in "editable" mode (importable directly
from source).

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

if REPO_GIT_URL:
    print(f"git clone {REPO_GIT_URL} -> {WORKDIR}")
    subprocess.run(["git", "clone", REPO_GIT_URL, WORKDIR], check=True)
else:
    print(f"extracting zip: {REPO_ZIP_ON_DRIVE} -> {WORKDIR}")
    with zipfile.ZipFile(REPO_ZIP_ON_DRIVE) as zf:
        zf.extractall("/content/_repo_zip_extract")
    # if the zip contains a single top-level folder (e.g. "my-bg-remover/"), move it to WORKDIR
    extracted = list(Path("/content/_repo_zip_extract").iterdir())
    src_root = extracted[0] if len(extracted) == 1 and extracted[0].is_dir() else Path("/content/_repo_zip_extract")
    if Path(WORKDIR).exists():
        shutil.rmtree(WORKDIR)
    shutil.move(str(src_root), WORKDIR)

%cd {WORKDIR}
!pip install -e . -q
print("Dependencies installed.")

## (c) Download the Training Sources in FULL

`data/train_sources.json` keeps a record of all training sources researched locally
in Phase 2 (where to download each one, its license, its estimated size). This cell
reads that file and downloads every source **in full** (not a small sample as was
done locally):

- **Hugging Face sources** (those with a non-empty `hf_repo` field: DIS5K-TR, CAMO-TR,
  P3M-10k, Transparent-460, BG-20k): downloaded via `huggingface_hub.snapshot_download`
  or by reading the parquet shards.
- **Google Drive sources** (those with a non-empty `drive_id` field: COD10K-TR always;
  HIM2K/AM-2k attempted if `DOWNLOAD_OPTIONAL_DRIVE_SOURCES` is on): downloaded with
  `gdown`. Drive access generally works in these environments (Colab); it was not
  possible in the local development environment (see the notes in `data/train_sources.json`).
- **Distinctions-646**: there is no publicly accessible download link (only by
  emailing the authors) — this source is ALWAYS skipped, only a note is printed.

The result is written to the SAME directory structure the local
`scripts/build_trainset.py` expects (`data/raw_train/<source>/{im,gt}` or
source-specific subfolders) — so the next cell (d) can call the same script
functions (just with full counts).

In [ ]:
!pip install gdown -q

import json
from pathlib import Path

RAW = Path("data/raw_train")
RAW.mkdir(parents=True, exist_ok=True)

with open("data/train_sources.json") as f:
    SOURCE_DEFS = {s["name"]: s for s in json.load(f)["sources"]}

print(f"{len(SOURCE_DEFS)} source definitions loaded: {list(SOURCE_DEFS)}")

In [ ]:
# --- Hugging Face sources: FULL splits are downloaded (via the parquet shards) ---
import io
import re

import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from PIL import Image

fs = HfFileSystem()


def _sanitize_stem(name) -> str:
    """Converts the source file name from the parquet into a safe stem (drop the
    extension, replace filesystem-unfriendly characters with '_')."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(str(name)).stem)


def _download_hf_parquet_pairs(source_name: str, img_col: str, mask_col: str, out_subdir: str) -> int:
    """Reads ALL parquet shards (split_patterns) of source_name row by row and
    writes the (image, mask) pairs under out_subdir/{im,gt}/. Memory friendly: each
    shard is read separately; the whole dataset is never loaded into memory at once.

    Stem strategy (COLLISION PREVENTION): if the schema has an `image_name` column,
    the stem is derived from it (the source dataset's own unique file names). If
    not, a CUMULATIVE counter increasing across ALL parquet shards is used — an
    index that resets per shard would, from the 2nd shard onward, collide with the
    1st shard's stems and cause those rows to be silently skipped as 'already
    downloaded' (e.g. only ~1/6 of the 6-shard DIS5K-TR would be downloaded)."""
    spec = SOURCE_DEFS[source_name]
    repo = spec["hf_repo"]
    out_im = RAW / out_subdir / "im"
    out_gt = RAW / out_subdir / "gt"
    out_im.mkdir(parents=True, exist_ok=True)
    out_gt.mkdir(parents=True, exist_ok=True)

    def _bytes_of(cell_value):
        # In the HF 'Image' feature parquet export the value usually arrives as a
        # {"bytes": ..., "path": ...} struct; in some mirrors it may be raw bytes directly.
        return cell_value["bytes"] if isinstance(cell_value, dict) else cell_value

    written = 0
    counter = 0  # cumulative row counter — NOT reset at shard boundaries
    for pattern in spec["split_patterns"]:
        paths = fs.glob(f"datasets/{repo}/{pattern}")
        for p in sorted(paths):
            print(f"  reading: {p}")
            with fs.open(p, "rb") as fh:
                schema_names = pq.read_schema(fh).names
                fh.seek(0)
                has_name = "image_name" in schema_names
                columns = (["image_name"] if has_name else []) + [img_col, mask_col]
                table = pq.read_table(fh, columns=columns)
            for i in range(table.num_rows):
                if has_name:
                    stem = f"{source_name}_{_sanitize_stem(table['image_name'][i].as_py())}"
                else:
                    stem = f"{source_name}_{counter:06d}"
                counter += 1
                out_img_path = out_im / f"{stem}.jpg"
                out_gt_path = out_gt / f"{stem}.png"
                if out_img_path.exists() and out_gt_path.exists():
                    continue  # idempotent: already downloaded (sorted() -> shard order stable, stems deterministic)
                img_bytes = _bytes_of(table[img_col][i].as_py())
                mask_bytes = _bytes_of(table[mask_col][i].as_py())
                Image.open(io.BytesIO(img_bytes)).convert("RGB").save(out_img_path, quality=95)
                Image.open(io.BytesIO(mask_bytes)).convert("L").save(out_gt_path)
                written += 1

    # --- Post-download integrity check: pair count on disk vs. the expected full set size ---
    total_pairs = len(list(out_im.glob("*")))
    expected = spec.get("full_pair_count")
    print(f"{source_name}: {written} new pairs written; {total_pairs} total on disk (expected ~{expected})")
    if expected and total_pairs < 0.9 * expected:
        raise RuntimeError(
            f"{source_name}: only {total_pairs}/{expected} pairs on disk (<90%) — "
            f"could be a stem collision, a missing parquet shard, or a changed repo schema; "
            f"inspect the 'reading' logs above."
        )
    return written


# NOTE: The generic function above is sensitive to the column names in the parquet
# schema; for `nobg/DIS5K` and `nobg/camo` we use the column names verified against
# the Phase 2 local sampling code (the scripts/build_trainset.py docstring). If the
# HF repo schema has changed, inspect the `pq.read_schema(...)` output and update
# img_col/mask_col.
try:
    _download_hf_parquet_pairs("dis5k_tr", img_col="image", mask_col="label", out_subdir="dis5k")
except Exception as e:
    print(f"WARNING: could not download dis5k_tr ({e}); if data/raw_train/dis5k exists (local sample), it will be used.")

try:
    _download_hf_parquet_pairs("camo_tr", img_col="image", mask_col="mask", out_subdir="camo")
except Exception as e:
    print(f"WARNING: could not download camo_tr ({e}); if data/raw_train/camo exists (local sample), it will be used.")

In [ ]:
# --- P3M-10k (single zip) and Transparent-460 (folder based) — FULL download via snapshot_download ---
from huggingface_hub import hf_hub_download, snapshot_download

# P3M-10k: the entire dataset is a single zip (data/p3m10k.zip, ~5.8GB); we download
# it and extract the TRAIN subset (blurred_image + mask).
p3m_zip = hf_hub_download(repo_id=SOURCE_DEFS["p3m_10k_train"]["hf_repo"],
                           repo_type="dataset", filename="data/p3m10k.zip")
import zipfile

p3m_out_im = RAW / "p3m" / "im"
p3m_out_gt = RAW / "p3m" / "gt"
p3m_out_im.mkdir(parents=True, exist_ok=True)
p3m_out_gt.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(p3m_zip) as zf:
    names = [n for n in zf.namelist() if "/train/blurred_image/" in n or "/train/mask/" in n]
    for n in names:
        if n.endswith("/"):  # zip directory entries (no content, skip)
            continue
        target = (p3m_out_im if "/blurred_image/" in n else p3m_out_gt) / Path(n).name
        if target.exists():
            continue
        with zf.open(n) as src, open(target, "wb") as dst:
            dst.write(src.read())
print(f"p3m_10k_train: {len(list(p3m_out_im.iterdir()))} images -> {p3m_out_im.parent}")

# Transparent-460: Train/{fg,alpha}/ full split (410 pairs, original resolution).
import shutil

trans_dir = snapshot_download(repo_id=SOURCE_DEFS["transparent_460_train"]["hf_repo"],
                               repo_type="dataset", allow_patterns=["Train/*"])
trans_out = RAW / "trans460_train"
if trans_out.exists():
    shutil.rmtree(trans_out)
shutil.copytree(Path(trans_dir) / "Train" / "fg", trans_out / "fg")
shutil.copytree(Path(trans_dir) / "Train" / "alpha", trans_out / "alpha")
print(f"transparent_460_train: {len(list((trans_out / 'fg').iterdir()))} images -> {trans_out}")

In [ ]:
# --- Google Drive sources: download with gdown ---
import zipfile as _zf


def _gdown_extract(drive_id: str, out_dir: Path, label: str) -> bool:
    """Downloads a zip from the Drive id and extracts it into out_dir. Returns
    False on failure (does not stop the pipeline — the caller proceeds with a note)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path = out_dir.parent / f"{out_dir.name}.zip"
    try:
        import gdown

        gdown.download(id=drive_id, output=str(zip_path), quiet=False)
        with _zf.ZipFile(zip_path) as zf:
            zf.extractall(out_dir)
        print(f"{label}: downloaded and extracted -> {out_dir}")
        return True
    except Exception as e:
        print(f"WARNING: could not download {label} ({e}) — this source will be SKIPPED (task instruction: skip if not found/not accessible).")
        return False


# COD10K-TR: important for the camouflage goal, always attempted.
cod10k_ok = _gdown_extract(SOURCE_DEFS["cod10k_tr"]["drive_id"], RAW / "cod10k_raw", "COD10K-TR")

# HIM2K / AM-2k: optional (depends on the DOWNLOAD_OPTIONAL_DRIVE_SOURCES flag).
him2k_ok = am2k_ok = False
if DOWNLOAD_OPTIONAL_DRIVE_SOURCES:
    him2k_ok = _gdown_extract(SOURCE_DEFS["him2k"]["drive_id"], RAW / "him2k_raw", "HIM2K")
    am2k_ok = _gdown_extract(SOURCE_DEFS["am_2k"]["drive_id"], RAW / "am2k_raw", "AM-2k")
else:
    print("HIM2K/AM-2k skipped (DOWNLOAD_OPTIONAL_DRIVE_SOURCES=False).")

print(f"\nDistinctions-646: {SOURCE_DEFS['distinctions_646']['download_method']}")
print("-> ALWAYS skipped (no publicly accessible download link).")

print(f"\nSummary: COD10K-TR={cod10k_ok}, HIM2K={him2k_ok}, AM-2k={am2k_ok}")
print("NOTE: the actual folder structure inside the COD10K-TR/HIM2K/AM-2k zips may "
      "differ from the official distribution (sets hosted via Drive do not guarantee "
      "a standard schema the way HF does) — if the download succeeded, before mapping "
      "these folders with build_trainset in the next cell (d), check the actual "
      "subfolder names (image/ vs Image/ vs img/, GT/ vs mask/) with a command like "
      "`!find data/raw_train/cod10k_raw -maxdepth 3` and update the globs below accordingly.")

## (d) Build the Full Manifest (with the `build_trainset.py` logic)

Now we produce `data/train/manifest.jsonl` from the raw data we downloaded — an
"inventory" file holding each training sample's identity (id), image path, category,
and GT mask path (the same format is used project-wide, including for the test set,
see `benchmark/testset.py`).

The source definitions (glob patterns + category rules) are read from the
**`SOURCE_SPECS`** dict inside `scripts/build_trainset.py` — the SINGLE source of
truth shared by the local script and the notebook; this is how hand-copied globs
drifting apart over time is prevented. The small sample sizes used in the local run
(`LOCAL_SAMPLE_N`, a disk budget constraint) are NOT used here: we call the same
functions (`sample_source`, `sample_disvd_tokens`, `copy=True` mode — real files
instead of symlinks, because symlinks can break when moving to Drive) **with the
full set** (`n=None` -> all matching pairs), plus we add the COD10K-TR (and, if
present, HIM2K/AM-2k) sources coming from Google Drive.

**HIM2K note (updated):** the `_matte` suffix assumption turned out to be WRONG — HIM2K actually uses an instance-matting structure (`images/train/<stem>.jpg` + `alphas/train/<stem>/00.png,01.png,...` — multiple instance alphas per image). The cell below no longer ASSUMES a fixed glob/suffix: it walks the directories with `os.walk`, discovers the img/gt pairs by stem overlap (for COD10K), and merges HIM2K's instance alphas pixel-wise with `max`, writing them under `data/raw_train/him2k_merged/{im,gt}`. For a tested, status-reporting (status.json to Drive) version of the same discovery/merge logic see `training/colab_devam_hucresi.py` (`discover_cod10k`, `discover_him2k_dirs`, `merge_him2k`) — if a live Colab runtime is resuming from this point, pasting that file into a SINGLE cell after cell (c) and running it is safer than re-running the remaining (d)-(g) cells by hand (same logic + state tracking).

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, "scripts")
import build_trainset as bt  # noqa: E402
from benchmark.testset import append_entries  # noqa: E402

# SINGLE SOURCE OF TRUTH: the glob/category definitions are read from
# build_trainset.SOURCE_SPECS (NO hand copies — if the local script changes, the
# notebook stays in sync automatically). The only differences are n=None (full set)
# and copy=True (real files on Colab, not symlinks).
# Each entry is (name, img_glob, gt_glob, category, n, extra_kwargs) — extra_kwargs
# carries additional keyword arguments passed to sample_source via ** (e.g.
# gt_stem_suffix); an empty dict for most sources.
FULL_SOURCES = [
    (name, spec["img_glob"], spec["gt_glob"], spec["category"], None, {})
    for name, spec in bt.SOURCE_SPECS.items()
    if spec["category"] != "disvd_tokens"  # dis5ktr is handled below with sample_disvd_tokens
]


# --- DISCOVERY of the actual COD10K/HIM2K folder structure (instead of ASSUMING globs/suffixes by hand) ---
# Motivation: the internal structure of Google Drive zips is not guaranteed (see the
# cell (c) note); the previous version ASSUMED "Image/GT" and a "_matte" suffix —
# that assumption turned out wrong for HIM2K (actual structure: images/train +
# alphas/train/<stem>/ instance PNGs). For a status-reported/tested version of the
# same algorithm (matching by stem overlap + instance max-merge), see
# training/colab_devam_hucresi.py.
def _walk_dirs(root: Path, max_depth: int = 4) -> list[dict]:
    root = Path(root)
    out = []
    for dirpath, dirnames, filenames in os.walk(root):
        rel = Path(dirpath).relative_to(root)
        depth = 0 if str(rel) == "." else len(rel.parts)
        if depth >= max_depth:
            dirnames[:] = []
        jpgs = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg"))]
        pngs = [f for f in filenames if f.lower().endswith(".png")]
        out.append({"path": Path(dirpath), "jpg_stems": {Path(f).stem for f in jpgs},
                    "png_stems": {Path(f).stem for f in pngs}})
    return out


def discover_cod10k(raw_dir: Path):
    """Matches a directory containing many .jpg files with the .png directory that
    shares most of its stems (stem overlap = the primary signal; the Image/GT naming
    is only a tie-break between equally scored candidates)."""
    if not raw_dir.exists():
        return None
    dirs = _walk_dirs(raw_dir)
    img_c = [d for d in dirs if len(d["jpg_stems"]) >= 10]
    gt_c = [d for d in dirs if len(d["png_stems"]) >= 10]
    best, best_score = None, (-1, -1)
    for ic in img_c:
        for gc in gt_c:
            if ic["path"] == gc["path"]:
                continue
            overlap = len(ic["jpg_stems"] & gc["png_stems"])
            if overlap == 0:
                continue
            bonus = int("image" in ic["path"].name.lower()) + int("gt" in gc["path"].name.lower())
            score = (overlap, bonus)
            if score > best_score:
                best_score, best = score, (ic["path"], gc["path"], overlap)
    return best


def discover_him2k_dirs(raw_dir: Path):
    """Finds the images/train and alphas/train directories by name (HIM2K's known
    official layout); returns None if not found (the source is skipped, the general
    category is optional)."""
    if not raw_dir.exists():
        return None
    images_dir = alphas_dir = None
    for dirpath, _dn, _fn in os.walk(raw_dir):
        p = Path(dirpath)
        if p.name.lower() == "train" and p.parent.name.lower() == "images":
            images_dir = p
        if p.name.lower() == "train" and p.parent.name.lower() == "alphas":
            alphas_dir = p
    return (images_dir, alphas_dir) if images_dir and alphas_dir else None


def merge_him2k(images_dir: Path, alphas_dir: Path, out_root: Path) -> int:
    """For each image: if alphas_dir/<stem>/ is a directory (instance PNGs), merges
    them pixel-wise with max; if it is a flat file (<stem>.png/.jpg), uses it directly."""
    import shutil

    import numpy as np
    from PIL import Image

    out_im, out_gt = out_root / "im", out_root / "gt"
    out_im.mkdir(parents=True, exist_ok=True)
    out_gt.mkdir(parents=True, exist_ok=True)
    images = {p.stem: p for p in images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
    count = 0
    for stem, img_path in sorted(images.items()):
        inst_dir = alphas_dir / stem
        merged = None
        if inst_dir.is_dir():
            for ip in sorted(list(inst_dir.glob("*.png")) + list(inst_dir.glob("*.jpg"))):
                arr = np.asarray(Image.open(ip).convert("L"), dtype=np.uint8)
                merged = arr if merged is None else np.maximum(merged, arr)
        else:
            for ext in (".png", ".jpg", ".jpeg"):
                cand = alphas_dir / f"{stem}{ext}"
                if cand.exists():
                    merged = np.asarray(Image.open(cand).convert("L"), dtype=np.uint8)
                    break
        if merged is None:
            continue
        Image.fromarray(merged, mode="L").save(out_gt / f"{stem}.png")
        shutil.copy2(img_path, out_im / img_path.name)
        count += 1
    return count


# If COD10K-TR was downloaded, add it (the actual subfolders are found by the discovery above).
if cod10k_ok:
    _cod_info = discover_cod10k(Path("data/raw_train/cod10k_raw"))
    if _cod_info:
        _img_dir, _gt_dir, _overlap = _cod_info
        print(f"COD10K discovery: img={_img_dir} gt={_gt_dir} overlapping stems={_overlap}")
        FULL_SOURCES.append(
            (f"cod10ktr", f"{_img_dir.relative_to(bt.ROOT)}/*", f"{_gt_dir.relative_to(bt.ROOT)}/*",
             "camouflage", None, {})
        )
    else:
        print("WARNING: could not discover an img/gt directory pair for COD10K — this source is being SKIPPED.")

# If HIM2K was downloaded: merge the instance alphas and add it (actual structure: images/train +
# alphas/train/<stem>/ — the "_matte" suffix assumption is NO LONGER USED, see the note above).
if him2k_ok:
    _him_dirs = discover_him2k_dirs(Path("data/raw_train/him2k_raw"))
    if _him_dirs:
        _images_dir, _alphas_dir = _him_dirs
        _merged_out = Path("data/raw_train/him2k_merged")
        _n_merged = merge_him2k(_images_dir, _alphas_dir, _merged_out)
        print(f"HIM2K instance alphas merged: {_n_merged} pairs -> {_merged_out}")
        if _n_merged > 0:
            FULL_SOURCES.append(
                ("him2k", "data/raw_train/him2k_merged/im/*", "data/raw_train/him2k_merged/gt/*",
                 "general", None, {})
            )
    else:
        print("WARNING: could not discover the HIM2K images/alphas directories — this source is being SKIPPED "
              "(the general category is optional anyway).")

if am2k_ok:
    FULL_SOURCES.append(
        ("am2k", "data/raw_train/am2k_raw/**/original/*", "data/raw_train/am2k_raw/**/mask/*",
         "general", None, {})
    )

bt.OUT_IMG.mkdir(parents=True, exist_ok=True)
bt.OUT_GT.mkdir(parents=True, exist_ok=True)
total_rows = 0
for name, img_glob, gt_glob, category, n, extra_kwargs in FULL_SOURCES:
    rows = bt.sample_source(name, img_glob, gt_glob, category, n, copy=True, **extra_kwargs)
    if rows:
        append_entries(str(bt.MANIFEST), rows)
    total_rows += len(rows)
    print(f"{name} ({category}): {len(rows)} pairs added")

# DIS5K-TR: sampled from a single pool; the category is assigned from the file-name token (thin/complex).
dis_spec = bt.SOURCE_SPECS["dis5ktr"]
dis_rows = bt.sample_disvd_tokens("dis5ktr", dis_spec["img_glob"], dis_spec["gt_glob"],
                                   n=None, copy=True)
if dis_rows:
    append_entries(str(bt.MANIFEST), dis_rows)
total_rows += len(dis_rows)

print(f"\nTotal manifest rows (added in this run): {total_rows}")
print(f"Manifest: {bt.MANIFEST}")

## (e) Composite + Augmentation Generation (`make_composites.py`)

This step combines each training image with a background (compositing) and adds
light distortions (augmentation). The per-category multipliers
(`CATEGORY_MULTIPLIERS`, defined in the parameters cell) ensure the transparency
and camouflage categories are represented with a higher share of the training mix —
since the project's goal is strong performance in these two categories. In the
camouflage category **no compositing is applied** (augment only): because the
essence of camouflage is the texture/color harmony between the object and its
original background, pasting it onto a random background would destroy that signal
(see `bgr/compositing.py` and the Phase 2 plan Task 4 note).

First we download the background pool (`BG_POOL_SIZE` images from BG-20k), then we
call the `make_composites.run(...)` function (the SAME function tested on the local
sample). The `per_image` value is roughly back-computed from the
`TARGET_TOTAL_IMAGES` target (not exact — hitting an exact number is hard because
of the category multipliers; it is an approximate estimate).

In [ ]:
# --- Background pool: BG_POOL_SIZE images from BG-20k ---
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from PIL import Image
import io

bg_dir = Path("data/backgrounds")
bg_dir.mkdir(parents=True, exist_ok=True)
existing_bg = len(list(bg_dir.iterdir()))
if existing_bg >= BG_POOL_SIZE:
    print(f"data/backgrounds already contains {existing_bg} images (>= {BG_POOL_SIZE}); skipping the download.")
else:
    fs = HfFileSystem()
    bg_spec = SOURCE_DEFS["bg_20k"]
    pattern = bg_spec["split_patterns"][0]  # "data/train-*-of-00022.parquet"
    parts = sorted(fs.glob(f"datasets/{bg_spec['hf_repo']}/{pattern}"))
    written = existing_bg
    for part in parts:
        if written >= BG_POOL_SIZE:
            break
        with fs.open(part, "rb") as fh:
            table = pq.read_table(fh, columns=["image"])
        for i in range(table.num_rows):
            if written >= BG_POOL_SIZE:
                break
            out_path = bg_dir / f"bg20k_{written:06d}.jpg"
            if out_path.exists():
                written += 1
                continue
            img_bytes = table["image"][i].as_py()["bytes"]
            im = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            im.thumbnail((1024, 1024))
            im.save(out_path, format="JPEG", quality=88)
            written += 1
    print(f"data/backgrounds: {written} background images.")

In [ ]:
# --- per_image estimate: target total / (source row count * average multiplier) ---
import make_composites as mc  # scripts/ is already on sys.path (see cell d)

if CATEGORY_MULTIPLIERS:  # if None, the scripts/make_composites.py default is KEPT
    mc.CATEGORY_MULTIPLIER = dict(CATEGORY_MULTIPLIERS)  # apply only if explicitly overridden

from benchmark.testset import load_manifest

manifest_rows = [r for r in load_manifest(str(bt.MANIFEST)) if r.get("gt_alpha")]
avg_multiplier = sum(mc.multiplier(r["category"]) for r in manifest_rows) / max(len(manifest_rows), 1)
per_image = max(1, round(TARGET_TOTAL_IMAGES / (len(manifest_rows) * avg_multiplier))) if manifest_rows else 1
print(f"Source rows: {len(manifest_rows)}, average multiplier: {avg_multiplier:.2f}, "
      f"chosen per_image: {per_image} (estimated total: {len(manifest_rows) * avg_multiplier * per_image:.0f})")

counts = mc.run(
    manifest_path=bt.MANIFEST,
    backgrounds_dir=bg_dir,
    per_image=per_image,
    seed=SEED,
    out_dir="data/train_composites",
)
print("Composites generated per category:", counts)

## (f) Export to BiRefNet Format (`export_birefnet.py`)

Final step: we convert `data/train_composites/manifest.jsonl` into the layout
expected by BiRefNet's official training code (`TRAIN/im/*.jpg` + `TRAIN/gt/*.png`,
pairs matched by identical file names); we also write `stats.json` (total count,
category distribution, resolution percentiles, per-category "soft alpha" ratio —
the share of partially transparent pixels in the transparency/matting sets).

In [ ]:
import json

import export_birefnet as eb  # scripts/ is already on sys.path

stats = eb.export(
    manifest_path="data/train_composites/manifest.jsonl",
    out_dir="/content/birefnet_format",
    split_name="TRAIN",
)
print(json.dumps(stats, ensure_ascii=False, indent=2))

## (g) Copy to Drive + Integrity Check

Finally we copy the `/content/birefnet_format` folder (Colab's temporary disk —
deleted when the session ends) to the persistent location on Drive
(`bg-remover-data/`). After copying, we verify that the file counts (source vs.
destination, `im/` vs `gt/`) match exactly and are consistent with the total count
in `stats.json` — this check matters because Drive synchronization can sometimes
be cut short on large folders.

In [ ]:
import json
import shutil
from pathlib import Path

src = Path("/content/birefnet_format")
dst = Path(DRIVE_ROOT) / DRIVE_OUTPUT_SUBDIR

print(f"Copying: {src} -> {dst}")
shutil.copytree(src, dst, dirs_exist_ok=True)

# --- Integrity check ---
src_im = list((src / "TRAIN" / "im").iterdir())
src_gt = list((src / "TRAIN" / "gt").iterdir())
dst_im = list((dst / "TRAIN" / "im").iterdir())
dst_gt = list((dst / "TRAIN" / "gt").iterdir())

with open(src / "stats.json") as f:
    stats_on_disk = json.load(f)

print(f"im/: source={len(src_im)}, destination={len(dst_im)}")
print(f"gt/: source={len(src_gt)}, destination={len(dst_gt)}")
print(f"stats.json total: {stats_on_disk['total']}")

assert len(src_im) == len(dst_im), "im/ file count does not match in the Drive copy!"
assert len(src_gt) == len(dst_gt), "gt/ file count does not match in the Drive copy!"
assert len(dst_im) == len(dst_gt) == stats_on_disk["total"], "im/gt/stats.json total counts are inconsistent!"

print("\nINTEGRITY CHECK PASSED — the data is ready on Drive.")
print(json.dumps(stats_on_disk, ensure_ascii=False, indent=2))